# VR Position Decoding — Bayesian Poisson Decoder & ML Ablation

## Decoder

**Bayesian Poisson** — argmax Poisson log-likelihood over smoothed tuning curves estimated
from training trials.  Tuning = mean spike count per 2 cm position-bin visit.

## Cells used

MEC **principal cells** only: `ENTm*` + mean VR firing rate < 10 Hz
(excludes fast-spiking interneurons; includes GC, NG, and Other classification groups).

## Cross-validation

Even-indexed trials → training tuning curves.  Odd-indexed trials → test.

Decoding error: circular distance `min(|decoded − actual|, 200 − |decoded − actual|)`,
where 0 and 200 cm are the same point on the track.

In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.stats import spearmanr, mannwhitneyu
from sklearn.model_selection import KFold
import time
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '/Users/harryclark/Documents/spatial-manifolds/src')
import pynapple as nap
from spatial_manifolds.detect_grids import curate_clusters

In [28]:
source_path         = '/Users/harryclark/Downloads/COHORT12/'
CLASSIFICATIONS_CSV     = '/Users/harryclark/Documents/spatial-manifolds/data/cell_classifications.csv'
VIS_CLASSIFICATIONS_CSV = '/Users/harryclark/Documents/spatial-manifolds/data/cell_classifications_no_regions.csv'
DECODE_CACHE        = '/Users/harryclark/Documents/spatial-manifolds/data/eddie/vr_decode_results.parquet'
ABLATION_CACHE      = '/Users/harryclark/Documents/spatial-manifolds/data/eddie/vr_decode_ablation.parquet'
COND_TRAIN_CACHE    = '/Users/harryclark/Documents/spatial-manifolds/data/eddie/vr_decode_cond_train.parquet'
VIS_DECODE_CACHE    = '/Users/harryclark/Documents/spatial-manifolds/data/eddie/vr_decode_vis_results.parquet'

TRACK_LEN = 200.0    # cm
N_BINS    = 40       # 5 cm per bin
SIGMA     = 1.5      # Gaussian smoothing for tuning-curve decoders (bins)
BIN_SIZE  = TRACK_LEN / N_BINS
BIN_EDGES   = np.linspace(0, TRACK_LEN, N_BINS + 1)
BIN_CENTRES = 0.5 * (BIN_EDGES[:-1] + BIN_EDGES[1:])
MIN_OCC           = 3     # min samples per position bin to include an observation
MIN_MEC_CELLS     = 5     # skip sessions with fewer MEC principal cells
MIN_VIS_CELLS     = 5     # skip sessions with fewer VIS principal cells
PRINCIPAL_MAX_FR  = 10.0  # Hz — excludes putative fast-spiking interneurons

ABLATION_DECODERS = ['bayesian_poisson']

# Multi-shank filter for ablation: only include sessions where the ML spread
# of recorded cells exceeds this threshold (µm). Single-shank sessions record
# all cells at roughly the same ML position, making medial→lateral ablation
# uninterpretable.
MIN_ML_SPREAD_UM = 200   # µm — roughly one shank spacing on a multi-shank probe

CHANCE_CM = TRACK_LEN / 4   # 50 cm — expected error for a uniform random decoder

# Minimum total number of trials a session must have to be included.
# With even/odd split this gives at least MIN_TRIALS//2 train and test trials.
MIN_TRIALS = 50

# 10-fold cross-validation for main decoding and condition-specificity analysis.
# Each fold trains on 90% of (condition) trials and tests on the held-out 10%.
# The ablation uses the simpler even/odd split to avoid 10x computational cost.
N_CV_FOLDS = 10

# Minimum training observations per condition per fold.
MIN_COND_TRAIN_OBS = 10

# Training conditions for the condition-specificity analysis.
# Each entry: (label, trial_type filter, performance filter).
# None means no filter (use all training observations).
TRAIN_CONDITIONS = [
    ('all',    None, None),
    ('b_hit',  'b',  'hit'),
    ('b_run',  'b',  'run'),
    ('nb_hit', 'nb', 'hit'),
    ('nb_run', 'nb', 'run'),
]

# Exemplar session shown in detail before the population analysis
EXEMPLAR_MOUSE, EXEMPLAR_DAY = 25, 25

mouse_days = {
    20: [14,15,16,17,18,19,20,21,22,23,24,25,26],
    21: [15,16,17,18,19,20,21,22,23,24,25,26],
    22: [33,34,35,36,37,38,39,40,41],
    25: [16,17,18,19,20,21,22,23,24,25],
    26: [11,12,13,14,15,16,17,18,19],
    27: [16,17,18,19,20,21,22,23,24,26],
    28: [16,17,18,19,20,21,22,23,25],
    29: [16,17,18,19,20,21,22,23,25],
}

# Colours per condition (for track / summary plots)
TRIAL_GROUPS = [
    ('all',    None, None,   '#455A64'),
    ('b_hit',  'b',  'hit',  '#1565C0'),
    ('b_run',  'b',  'run',  '#42A5F5'),
    ('nb_hit', 'nb', 'hit',  '#C62828'),
    ('nb_run', 'nb', 'run',  '#EF9A9A'),
]

DECODER_COLOUR = '#1B5E20'   # MEC bayesian_poisson
VIS_COLOUR     = '#6A1B9A'   # VIS bayesian_poisson

In [29]:
# ══════════════════════════════════════════════════════════════════════════════
# Core utilities
# ══════════════════════════════════════════════════════════════════════════════

def circular_error_cm(actual, decoded, track_len=TRACK_LEN):
    """Circular absolute error (cm). 0 and track_len are equivalent."""
    diff = np.abs(np.asarray(decoded) - np.asarray(actual))
    return np.minimum(diff, track_len - diff)


def load_vr_session(mouse, day, src):
    sf = f'{src}M{mouse}/D{day:02}/VR/'
    beh      = nap.load_file(sf + f'sub-{mouse}_day-{day:02}_ses-VR_beh.nwb')
    clusters = curate_clusters(
        nap.load_file(sf + f'sub-{mouse}_day-{day:02}_ses-VR_srt-kilosort4_clusters.npz'))
    return beh, clusters


def build_obs_matrix(mec_cids, clusters, pos_t, pos_v, trial_indices, all_trials):
    """
    Spike-count feature matrix for the given trials.

    Returns
    -------
    X    : (N_obs, N_cells)  spike counts per position-bin visit
    pos  : (N_obs,)          actual position (bin centre, cm)
    meta : list[dict]        trial_type, performance per observation
    """
    dt         = float(np.nanmedian(np.diff(pos_t)))
    spk_trains = [np.asarray(clusters[cid].t) for cid in mec_cids]
    X_list, pos_list, meta_list = [], [], []

    for ti in trial_indices:
        t0, t1 = float(all_trials.start[ti]), float(all_trials.end[ti])
        ttype  = str(all_trials.type[ti])
        perf   = str(all_trials.performance[ti])
        pm     = (pos_t >= t0) & (pos_t <= t1)
        tp_v   = pos_v[pm]; tp_t = pos_t[pm]
        if pm.sum() < MIN_OCC * 2:
            continue
        for bi in range(N_BINS):
            in_bin = (tp_v >= BIN_EDGES[bi]) & (tp_v < BIN_EDGES[bi + 1])
            if in_bin.sum() < MIN_OCC:
                continue
            t_pts = tp_t[in_bin]
            spk   = np.array(
                [((s >= t_pts[0]) & (s <= t_pts[-1])).sum() for s in spk_trains],
                dtype=np.float32)
            X_list.append(spk)
            pos_list.append(BIN_CENTRES[bi])
            meta_list.append(dict(trial_type=ttype, performance=perf))

    if not X_list:
        return None, None, None
    return np.vstack(X_list), np.array(pos_list), meta_list


def _tuning_from_obs(X_train, pos_train, sigma=SIGMA):
    """Estimate (N_cells, N_bins) smoothed tuning curves from observation matrix."""
    n_cells = X_train.shape[1]
    tuning  = np.zeros((n_cells, N_BINS))
    for bi in range(N_BINS):
        mask = (pos_train >= BIN_EDGES[bi]) & (pos_train < BIN_EDGES[bi + 1])
        if mask.sum() > 0:
            tuning[:, bi] = X_train[mask].mean(axis=0)
    return gaussian_filter1d(tuning, sigma=sigma, axis=1)

In [30]:
def bayesian_poisson(X_train, pos_train, X_test):
    """Argmax Poisson log-likelihood over smoothed tuning curves."""
    tuning  = _tuning_from_obs(X_train, pos_train)          # (N_cells, N_bins)
    log_tc  = np.log(tuning + 1e-10)
    log_lik = X_test @ log_tc - tuning.sum(axis=0)[None, :] # (N_test, N_bins)
    return BIN_CENTRES[np.argmax(log_lik, axis=1)]


DECODERS = {'bayesian_poisson': bayesian_poisson}

In [ ]:
cache_path      = Path(DECODE_CACHE)
abl_path        = Path(ABLATION_CACHE)
cond_train_path = Path(COND_TRAIN_CACHE)
cache_path.parent.mkdir(parents=True, exist_ok=True)

# Check whether cached files cover all current decoders / conditions
def _cache_valid(path, col, required):
    if not path.exists(): return False
    df = pd.read_parquet(path)
    return col in df.columns and set(required).issubset(df[col].unique())

decode_ok     = _cache_valid(cache_path,      'decoder',         DECODERS.keys())
ablate_ok     = _cache_valid(abl_path,        'decoder',         ABLATION_DECODERS)
cond_train_ok = _cache_valid(cond_train_path, 'train_condition',
                              [tc for tc, *_ in TRAIN_CONDITIONS])

def _filter_decoders(df, col='decoder'):
    return df[df[col].isin(DECODERS.keys())].reset_index(drop=True)

if decode_ok and ablate_ok and cond_train_ok:
    print('Loading all cached results...')
    df_decode      = _filter_decoders(pd.read_parquet(DECODE_CACHE))
    df_ablation    = _filter_decoders(pd.read_parquet(ABLATION_CACHE))
    df_cond_train  = pd.read_parquet(COND_TRAIN_CACHE)
else:
    if decode_ok and not ablate_ok:
        print('Decode cache valid; rerunning ablation + condition training only...')
        df_decode = _filter_decoders(pd.read_parquet(DECODE_CACHE))
        rebuild_decode = False
    else:
        print('Rebuilding all results...')
        rebuild_decode = True

    df_class = pd.read_csv(CLASSIFICATIONS_CSV)[
        ['mouse','day','cluster_id','brain_region','firing_rate_VR','SC_x']
    ].copy()
    df_class['mouse'] = df_class['mouse'].astype(int)
    df_class['day']   = df_class['day'].astype(int)
    df_class['ml']    = df_class['SC_x'].abs()

    decode_rows      = [] if rebuild_decode else None
    ablation_rows    = []
    cond_train_rows  = []

    for mouse, days in mouse_days.items():
        for day in days:
            try:
                beh, clusters = load_vr_session(mouse, day, source_path)
            except Exception as e:
                print(f'  skip M{mouse}D{day}: {e}')
                continue

            # ── MEC principal cells ──────────────────────────────────────
            sc = df_class[
                (df_class['mouse'] == int(mouse)) &
                (df_class['day']   == int(day))
            ].set_index('cluster_id')

            mec_cids = [
                cid for cid in clusters.index
                if cid in sc.index
                and str(sc.loc[cid, 'brain_region']).startswith('ENTm')
                and float(sc.loc[cid, 'firing_rate_VR']) < PRINCIPAL_MAX_FR
                and not np.isnan(float(sc.loc[cid, 'SC_x']))
            ]
            if len(mec_cids) < MIN_MEC_CELLS:
                continue

            ml_vals    = np.array([abs(float(sc.loc[cid, 'SC_x'])) for cid in mec_cids])
            all_trials = beh['trials']
            n_trials   = len(all_trials)
            if n_trials < MIN_TRIALS:
                continue

            pos_t   = np.asarray(beh['P'].t)
            pos_v   = np.asarray(beh['P'].values)
            n_cells = len(mec_cids)

            # ── 10-fold CV: decode + condition-specificity ────────────────
            # Each fold: train on 90% of trials, test on held-out 10%.
            # All trials contribute a test prediction → no data wasted.
            kf = KFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=42)
            trial_ids = np.arange(n_trials)

            for fold_idx, (tr_idx, te_idx) in enumerate(kf.split(trial_ids)):
                X_tr, pos_tr, meta_tr = build_obs_matrix(
                    mec_cids, clusters, pos_t, pos_v, list(tr_idx), all_trials)
                X_te, pos_te, meta_te = build_obs_matrix(
                    mec_cids, clusters, pos_t, pos_v, list(te_idx), all_trials)

                if X_tr is None or X_te is None:
                    continue

                # Standard all-trial decoder
                if rebuild_decode:
                    try:
                        ph  = bayesian_poisson(X_tr, pos_tr, X_te)
                        err = circular_error_cm(pos_te, ph)
                        for obs_i, m in enumerate(meta_te):
                            decode_rows.append(dict(
                                mouse=int(mouse), day=int(day),
                                n_cells=n_cells, decoder='bayesian_poisson',
                                fold=fold_idx,
                                trial_type=m['trial_type'], performance=m['performance'],
                                actual_pos=pos_te[obs_i],
                                decoded_pos=float(ph[obs_i]),
                                error_cm=float(err[obs_i]),
                            ))
                    except Exception as e:
                        print(f'    decode fold {fold_idx} failed: {e}')

                # Condition-specific training
                for train_cond, train_tt, train_perf in TRAIN_CONDITIONS:
                    if train_tt is None:
                        cmask = np.ones(len(meta_tr), dtype=bool)
                    else:
                        cmask = np.array(
                            [m['trial_type'] == train_tt and m['performance'] == train_perf
                             for m in meta_tr], dtype=bool)
                    if cmask.sum() < MIN_COND_TRAIN_OBS:
                        continue
                    try:
                        ph  = bayesian_poisson(X_tr[cmask], pos_tr[cmask], X_te)
                        err = circular_error_cm(pos_te, ph)
                        for obs_i, m in enumerate(meta_te):
                            cond_train_rows.append(dict(
                                mouse=int(mouse), day=int(day),
                                n_cells=n_cells, fold=fold_idx,
                                train_condition=train_cond,
                                trial_type=m['trial_type'], performance=m['performance'],
                                actual_pos=pos_te[obs_i],
                                decoded_pos=float(ph[obs_i]),
                                error_cm=float(err[obs_i]),
                            ))
                    except Exception as e:
                        print(f'    cond {train_cond} fold {fold_idx} failed: {e}')

            # ── Ablation: even/odd split (10-fold too slow per step) ──────
            idx_train_abl = list(range(0, n_trials, 2))
            idx_test_abl  = list(range(1, n_trials, 2))
            X_train_abl, pos_train_abl, _ = build_obs_matrix(
                mec_cids, clusters, pos_t, pos_v, idx_train_abl, all_trials)
            X_test_abl,  pos_test_abl,  meta_test_abl = build_obs_matrix(
                mec_cids, clusters, pos_t, pos_v, idx_test_abl,  all_trials)

            if X_train_abl is None or X_test_abl is None:
                print(f'  M{mouse}D{day}: {n_cells} cells | done')
                continue

            ml_spread = ml_vals.max() - ml_vals.min()
            if ml_spread < MIN_ML_SPREAD_UM:
                print(f'  M{mouse}D{day}: {n_cells} cells | done')
                continue

            order_med = np.argsort(ml_vals)[::-1]
            order_lat = np.argsort(ml_vals)
            step      = max(1, n_cells // 20)

            abl_ttype = np.array([m['trial_type'] for m in meta_test_abl])
            abl_perf  = np.array([m['performance'] for m in meta_test_abl])
            ABL_CONDS = [
                ('all',    None, None),
                ('b_hit',  'b',  'hit'),
                ('b_run',  'b',  'run'),
                ('nb_hit', 'nb', 'hit'),
                ('nb_run', 'nb', 'run'),
            ]

            for dec_name in ABLATION_DECODERS:
                dec_fn = DECODERS[dec_name]
                for direction, order in [('medial_first',  order_med),
                                         ('lateral_first', order_lat)]:
                    for n_remove in range(0, n_cells, step):
                        keep = order[n_remove:]
                        if len(keep) == 0:
                            continue
                        try:
                            ph     = dec_fn(X_train_abl[:, keep], pos_train_abl,
                                            X_test_abl[:, keep])
                            errors = circular_error_cm(pos_test_abl, ph)
                        except Exception:
                            continue
                        base = dict(mouse=int(mouse), day=int(day),
                                    decoder=dec_name, direction=direction,
                                    n_cells_total=n_cells, n_cells_kept=len(keep),
                                    frac_kept=len(keep) / n_cells)
                        for cond_key, tt, pp in ABL_CONDS:
                            if tt is None:
                                amask = np.ones(len(errors), dtype=bool)
                            else:
                                amask = (abl_ttype == tt) & (abl_perf == pp)
                            if amask.sum() < 3:
                                continue
                            ablation_rows.append({**base,
                                'condition': cond_key,
                                'mean_error_cm': float(errors[amask].mean())})

            print(f'  M{mouse}D{day}: {n_cells} cells | done')

    if rebuild_decode:
        df_decode = pd.DataFrame(decode_rows)
        df_decode.to_parquet(DECODE_CACHE, index=False)
    df_ablation   = pd.DataFrame(ablation_rows)
    df_cond_train = pd.DataFrame(cond_train_rows)
    df_ablation.to_parquet(ABLATION_CACHE, index=False)
    df_cond_train.to_parquet(COND_TRAIN_CACHE, index=False)
    print('Saved.')

print(f'\nDecode rows        : {len(df_decode):,}')
print(f'Cond-train rows    : {len(df_cond_train):,}')
print(f'Ablation rows      : {len(df_ablation):,}')
if len(df_cond_train):
    print('\nTrain conditions found:', df_cond_train["train_condition"].unique().tolist())

# ── VIS principal cell decode ─────────────────────────────────────────────
# Same bayesian_poisson decoder, same 10-fold CV, but using VIS (visual
# cortex) cells as a region comparison for the track-error plot.
vis_cache_path = Path(VIS_DECODE_CACHE)
vis_cache_ok   = _cache_valid(vis_cache_path, 'decoder', DECODERS.keys())

if vis_cache_ok:
    df_vis = _filter_decoders(pd.read_parquet(VIS_DECODE_CACHE))
    print(f'VIS cache loaded: {len(df_vis):,} rows')
else:
    print('Building VIS decode cache...')
    _df_cls_v = pd.read_csv(VIS_CLASSIFICATIONS_CSV)[
        ['mouse','day','cluster_id','brain_region','firing_rate_VR']
    ].copy()
    _df_cls_v['mouse'] = _df_cls_v['mouse'].astype(int)
    _df_cls_v['day']   = _df_cls_v['day'].astype(int)
    vis_rows = []
    for _vm, _vdays in mouse_days.items():
        for _vd in _vdays:
            try:
                _vbeh, _vclusters = load_vr_session(_vm, _vd, source_path)
            except Exception as _e:
                print(f'  skip M{_vm}D{_vd}: {_e}'); continue
            _vsc = _df_cls_v[(_df_cls_v['mouse']==int(_vm)) &
                              (_df_cls_v['day']==int(_vd))].set_index('cluster_id')
            _vis_cids = [
                cid for cid in _vclusters.index
                if cid in _vsc.index
                and 'VIS' in str(_vsc.loc[cid,'brain_region'])
                and float(_vsc.loc[cid,'firing_rate_VR']) < PRINCIPAL_MAX_FR
            ]
            if len(_vis_cids) < MIN_VIS_CELLS:
                continue
            _vat = _vbeh['trials']
            if len(_vat) < MIN_TRIALS:
                continue
            _vpt = np.asarray(_vbeh['P'].t)
            _vpv = np.asarray(_vbeh['P'].values)
            _vkf = KFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=42)
            for _vfi, (_vtri, _vtei) in enumerate(_vkf.split(np.arange(len(_vat)))):
                _vXtr, _vptr, _ = build_obs_matrix(
                    _vis_cids, _vclusters, _vpt, _vpv, list(_vtri), _vat)
                _vXte, _vpte, _vmte = build_obs_matrix(
                    _vis_cids, _vclusters, _vpt, _vpv, list(_vtei), _vat)
                if _vXtr is None or _vXte is None:
                    continue
                try:
                    _vph  = bayesian_poisson(_vXtr, _vptr, _vXte)
                    _verr = circular_error_cm(_vpte, _vph)
                    for _vi, _vm2 in enumerate(_vmte):
                        vis_rows.append(dict(
                            mouse=int(_vm), day=int(_vd),
                            n_cells=len(_vis_cids), decoder='bayesian_poisson',
                            fold=_vfi,
                            trial_type=_vm2['trial_type'], performance=_vm2['performance'],
                            actual_pos=_vpte[_vi], decoded_pos=float(_vph[_vi]),
                            error_cm=float(_verr[_vi])))
                except Exception:
                    continue
            print(f'  VIS M{_vm}D{_vd}: {len(_vis_cids)} cells | done')
    df_vis = pd.DataFrame(vis_rows)
    vis_cache_path.parent.mkdir(parents=True, exist_ok=True)
    df_vis.to_parquet(VIS_DECODE_CACHE, index=False)
    print(f'VIS decode saved: {len(df_vis):,} rows')

print(f'VIS decode rows    : {len(df_vis):,}')

Loading all cached results...

Decode rows        : 385,463
Cond-train rows    : 1,632,299
Ablation rows      : 4,966

Train conditions found: ['all', 'b_hit', 'b_run', 'nb_hit', 'nb_run']
Building VIS decode cache...
  VIS M21D19: 115 cells | done
  VIS M21D20: 71 cells | done
  VIS M21D21: 62 cells | done
  VIS M21D23: 92 cells | done
  VIS M21D24: 35 cells | done
  VIS M21D25: 59 cells | done
  VIS M21D26: 38 cells | done
  VIS M22D33: 26 cells | done
  VIS M22D34: 46 cells | done
  VIS M22D35: 17 cells | done
  VIS M22D36: 16 cells | done
  VIS M22D39: 250 cells | done
  VIS M22D40: 247 cells | done
  VIS M22D41: 197 cells | done
  VIS M25D22: 16 cells | done
  VIS M25D23: 7 cells | done
  VIS M25D25: 24 cells | done
  VIS M26D11: 22 cells | done
  VIS M26D12: 28 cells | done
  VIS M26D14: 5 cells | done
  VIS M26D15: 12 cells | done
  VIS M26D16: 32 cells | done
  VIS M26D17: 13 cells | done


---
## Exemplar session — M{EXEMPLAR_MOUSE}D{EXEMPLAR_DAY}

A single session is decoded in detail before the population analysis.
This cell is self-contained and can be run independently of `cell-collect`.

In [ ]:
# ── Load exemplar session ─────────────────────────────────────────────────────
beh_ex, clusters_ex = load_vr_session(EXEMPLAR_MOUSE, EXEMPLAR_DAY, source_path)

_clf_ex = pd.read_csv(CLASSIFICATIONS_CSV,
                      usecols=['mouse','day','cluster_id',
                                'brain_region','firing_rate_VR','SC_x'])
sc_ex = _clf_ex[
    (_clf_ex['mouse'].astype(int) == int(EXEMPLAR_MOUSE)) &
    (_clf_ex['day'].astype(int)   == int(EXEMPLAR_DAY))
].set_index('cluster_id')

mec_cids_ex = [
    cid for cid in clusters_ex.index
    if cid in sc_ex.index
    and str(sc_ex.loc[cid, 'brain_region']).startswith('ENTm')
    and float(sc_ex.loc[cid, 'firing_rate_VR']) < PRINCIPAL_MAX_FR
    and not np.isnan(float(sc_ex.loc[cid, 'SC_x']))
]

all_trials_ex = beh_ex['trials']
n_trials_ex   = len(all_trials_ex)
idx_train_ex  = list(range(0, n_trials_ex, 2))
idx_test_ex   = list(range(1, n_trials_ex, 2))
pos_t_ex      = np.asarray(beh_ex['P'].t)
pos_v_ex      = np.asarray(beh_ex['P'].values)

X_train_ex, pos_train_ex, _ = build_obs_matrix(
    mec_cids_ex, clusters_ex, pos_t_ex, pos_v_ex, idx_train_ex, all_trials_ex)
X_test_ex, pos_test_ex, meta_ex = build_obs_matrix(
    mec_cids_ex, clusters_ex, pos_t_ex, pos_v_ex, idx_test_ex, all_trials_ex)

n_cells_ex = len(mec_cids_ex)
print(f'M{EXEMPLAR_MOUSE}D{EXEMPLAR_DAY}: {n_cells_ex} MEC principal cells, '
      f'{n_trials_ex} trials ({len(idx_train_ex)} train / {len(idx_test_ex)} test), '
      f'{len(X_test_ex)} test observations')

# ── Figure 1: Tuning curve heatmap + best-decoder scatter ─────────────────────
tuning_ex = _tuning_from_obs(X_train_ex, pos_train_ex)     # (N_cells, N_bins)
t_norm    = tuning_ex / (tuning_ex.max(axis=1, keepdims=True) + 1e-10)
sort_idx  = np.argsort(tuning_ex.argmax(axis=1))
t_sorted  = t_norm[sort_idx]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), gridspec_kw=dict(wspace=0.40))

ax = axes[0]
im = ax.imshow(t_sorted, aspect='auto', cmap='viridis', interpolation='nearest',
               extent=[0, TRACK_LEN, n_cells_ex, 0])
plt.colorbar(im, ax=ax, label='Normalised rate', pad=0.02)
ax.set_xlabel('Position (cm)', fontsize=9)
ax.set_ylabel('Cell (sorted by preferred position)', fontsize=9)
ax.set_title(f'M{EXEMPLAR_MOUSE}D{EXEMPLAR_DAY} — MEC tuning curves (n={n_cells_ex})',
             fontsize=9, fontweight='bold')
ax.tick_params(labelsize=7)

# Bayesian Poisson as representative decoder
ph_ex  = bayesian_poisson(X_train_ex, pos_train_ex, X_test_ex)
err_ex = circular_error_cm(pos_test_ex, ph_ex)

ax = axes[1]
sc = ax.scatter(pos_test_ex, ph_ex, c=pos_test_ex, cmap='hsv',
                s=6, alpha=0.35, vmin=0, vmax=TRACK_LEN)
ax.plot([0, TRACK_LEN], [0, TRACK_LEN], 'k--', lw=0.8, label='Perfect decode')
plt.colorbar(sc, ax=ax, label='Actual position (cm)', pad=0.02)
ax.set_xlabel('Actual position (cm)', fontsize=9)
ax.set_ylabel('Decoded position (cm)', fontsize=9)
ax.set_title(f'Bayesian Poisson decoder — Median error = {np.median(err_ex):.1f} cm  (chance = {TRACK_LEN/4:.0f} cm)', fontsize=9, fontweight='bold')
ax.set_xlim(0, TRACK_LEN); ax.set_ylim(0, TRACK_LEN)
ax.legend(fontsize=7.5, frameon=False)
ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=7)

fig.suptitle(f'M{EXEMPLAR_MOUSE}D{EXEMPLAR_DAY} — tuning curves & example decode',
             fontsize=10, fontweight='bold')
plt.savefig(f'vr_exemplar_M{EXEMPLAR_MOUSE}D{EXEMPLAR_DAY}_tuning.pdf',
            bbox_inches='tight', dpi=150)
plt.show()



---
## Decoding pipeline — example sessions

Position raster (colour = trial type), trial-type × performance splits, example MEC tuning
curves per condition (parula), and decoded-position heatmap (jet, colour = decoded cm).
Shown for all sessions with ≥ 10 trials in every condition.

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from spatial_manifolds.anaylsis_parameters import cmap_parula
from spatial_manifolds.behaviour_plots import plot_firing_rate_map as _pfr
from scipy.ndimage import gaussian_filter as _gf

SCHEMATIC_SESSIONS = [
    (20, 15), (20, 16), (20, 17), (20, 18), (20, 23), (20, 24), (20, 26),
    (25, 16), (25, 24),
    (27, 18), (27, 23), (27, 26),
]

PLOT_CONDS = [
    ('b_hit',  'b',  'hit',  '#1565C0', 'Beaconed hit'),
    ('b_run',  'b',  'run',  '#42A5F5', 'Beaconed run'),
    ('nb_hit', 'nb', 'hit',  '#C62828', 'NB hit'),
    ('nb_run', 'nb', 'run',  '#EF9A9A', 'NB run'),
]

_SS = 3   # raster subsample
_clf_s = pd.read_csv(CLASSIFICATIONS_CSV,
                      usecols=['mouse','day','cluster_id',
                                'brain_region','firing_rate_VR','SC_x'])

def _get_mec(mouse, day, clusters):
    sc = _clf_s[(_clf_s['mouse'].astype(int)==mouse) &
                (_clf_s['day'].astype(int)==day)].set_index('cluster_id')
    return [cid for cid in clusters.index
            if cid in sc.index
            and str(sc.loc[cid,'brain_region']).startswith('ENTm')
            and float(sc.loc[cid,'firing_rate_VR']) < PRINCIPAL_MAX_FR
            and not np.isnan(float(sc.loc[cid,'SC_x']))]

def _raster(ax, trlist, colour, title=None, show_x=False):
    for _row, _t in enumerate(trlist):
        _p = _t['pos'][::_SS]
        if len(_p):
            ax.scatter(_p, np.full(len(_p), _row), s=0.8, c=colour,
                       alpha=0.22, lw=0, rasterized=True)
    ax.set_xlim(0, TRACK_LEN)
    ax.set_ylim(-0.5, max(len(trlist)-0.5, 0.5))
    ax.invert_yaxis()
    ax.set_yticks([])
    ax.spines[['top','right','left']].set_visible(False)
    ax.tick_params(axis='x', labelsize=6)
    if title:
        ax.set_title(title, fontsize=7.5, fontweight='bold', pad=2, color=colour)
    if not show_x:
        ax.set_xticklabels([])
    else:
        ax.set_xlabel('Position (cm)', fontsize=6)

def _trial_tc(cid, trial_list, clusters, beh, trials):
    _grp = nap.TsGroup({cid: clusters[cid]})
    _parts = []
    for _t in trial_list:
        _ep = nap.IntervalSet(
            start=float(trials.start[_t['ti']]),
            end=float(trials.end[_t['ti']]))
        _tc = nap.compute_1d_tuning_curves(
            _grp, beh['P'], nb_bins=N_BINS, minmax=[0, TRACK_LEN], ep=_ep)[cid]
        _parts.append(np.array(_tc))
    if not _parts:
        return None
    return _gf(np.nan_to_num(np.concatenate(_parts)).astype(np.float64), sigma=2.5)

# ── Loop over sessions ────────────────────────────────────────────────────
for _mouse, _day in SCHEMATIC_SESSIONS:
    print(f'\nM{_mouse}D{_day}', end=' ... ', flush=True)

    beh_s, clusters_s = load_vr_session(_mouse, _day, source_path)
    mec_s    = _get_mec(_mouse, _day, clusters_s)
    pos_t_s  = np.asarray(beh_s['P'].t)
    pos_v_s  = np.asarray(beh_s['P'].values)
    trials_s = beh_s['trials']
    n_t_s    = len(trials_s)

    if len(mec_s) < MIN_MEC_CELLS:
        print(f'skip ({len(mec_s)} MEC cells)')
        continue

    trial_info_s = []
    for _ti in range(n_t_s):
        _t0 = float(trials_s.start[_ti])
        _t1 = float(trials_s.end[_ti])
        _pm = (pos_t_s >= _t0) & (pos_t_s <= _t1)
        trial_info_s.append({'ti': _ti,
                              'ttype': str(trials_s.type[_ti]),
                              'perf':  str(trials_s.performance[_ti]),
                              'pos':   pos_v_s[_pm]})

    def _gc(ttype=None, perf=None):
        return [t for t in trial_info_s
                if (ttype is None or t['ttype']==ttype)
                and (perf  is None or t['perf']==perf)]

    # example cells — top-3 by all-trial peak/mean ratio
    _Xa, _pa, _ = build_obs_matrix(
        mec_s, clusters_s, pos_t_s, pos_v_s, list(range(n_t_s)), trials_s)
    if _Xa is None:
        print('skip (no obs)')
        continue
    _pmr  = _tuning_from_obs(_Xa, _pa).max(axis=1) /             (_tuning_from_obs(_Xa, _pa).mean(axis=1) + 1e-10)
    ex_s  = np.argsort(_pmr)[-3:][::-1]

    # decoded position heatmap (10-fold CV within each condition)
    _kf = KFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=42)
    dec_map_s = {}
    for _key, _tt, _pp, _col, _lbl in PLOT_CONDS:
        _cti = [t['ti'] for t in _gc(_tt, _pp)]
        if len(_cti) < N_CV_FOLDS:
            dec_map_s[_key] = None; continue
        _ds = np.zeros((len(_cti), N_BINS))
        _dc = np.zeros((len(_cti), N_BINS))
        for _tri, _tei in _kf.split(_cti):
            _Xtr, _ptr, _ = build_obs_matrix(
                mec_s, clusters_s, pos_t_s, pos_v_s, [_cti[i] for i in _tri], trials_s)
            if _Xtr is None or len(_Xtr) < MIN_COND_TRAIN_OBS:
                continue
            for _oi in _tei:
                _Xte, _pte, _ = build_obs_matrix(
                    mec_s, clusters_s, pos_t_s, pos_v_s, [_cti[_oi]], trials_s)
                if _Xte is None:
                    continue
                for _a, _d in zip(_pte, bayesian_poisson(_Xtr, _ptr, _Xte)):
                    _bi = min(int(np.digitize(_a, BIN_EDGES[1:-1])), N_BINS-1)
                    _ds[_oi, _bi] += _d; _dc[_oi, _bi] += 1
        dec_map_s[_key] = np.where(_dc > 0, _ds/_dc, np.nan)

    # ── Figure ────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(20, 8))
    gs  = gridspec.GridSpec(4, 7, figure=fig,
                             width_ratios=[2, 1.5, 1.5, 1.2, 1.2, 1.2, 1.8],
                             hspace=0.12, wspace=0.38)
    ax_all     = fig.add_subplot(gs[:, 0])
    ax_b       = fig.add_subplot(gs[0:2, 1])
    ax_nb      = fig.add_subplot(gs[2:4, 1])
    ax_cond    = [fig.add_subplot(gs[ri, 2]) for ri in range(4)]
    axes_cells = [[fig.add_subplot(gs[ri, 3+ci]) for ci in range(3)] for ri in range(4)]
    ax_dec     = [fig.add_subplot(gs[ri, 6]) for ri in range(4)]

    # col 0: all trials
    for _t in trial_info_s:
        _p = _t['pos'][::_SS]
        if len(_p):
            ax_all.scatter(_p, np.full(len(_p), _t['ti']),
                           s=0.8, c='#1565C0' if _t['ttype']=='b' else '#C62828',
                           alpha=0.18, lw=0, rasterized=True)
    ax_all.set_xlim(0, TRACK_LEN); ax_all.set_ylim(-0.5, n_t_s-0.5)
    ax_all.invert_yaxis(); ax_all.set_yticks([])
    ax_all.set_xlabel('Position (cm)', fontsize=7)
    ax_all.set_ylabel('Trial number', fontsize=7)
    ax_all.set_title(f'M{_mouse}D{_day} — all trials ({n_t_s})', fontsize=8, fontweight='bold')
    ax_all.spines[['top','right']].set_visible(False); ax_all.tick_params(labelsize=6)
    ax_all.legend(handles=[mpatches.Patch(color='#1565C0', label='Beaconed'),
                             mpatches.Patch(color='#C62828', label='Non-beaconed')],
                   fontsize=5.5, frameon=False, loc='upper right',
                   handlelength=0.8, handleheight=0.8)

    # col 1: beaconed / non-beaconed
    _raster(ax_b,  _gc('b'),  '#1565C0', title=f'Beaconed (n={len(_gc("b"))})')
    _raster(ax_nb, _gc('nb'), '#C62828', title=f'Non-beaconed (n={len(_gc("nb"))})', show_x=True)

    # col 2: per-condition rasters
    for ri, (_key, _tt, _pp, _col, _lbl) in enumerate(PLOT_CONDS):
        _raster(ax_cond[ri], _gc(_tt, _pp), _col,
                title=f'{_lbl} (n={len(_gc(_tt,_pp))})', show_x=(ri==3))

    # cols 3-5: rate-map heatmaps (parula)
    for ci in range(3):
        _cid = mec_s[ex_s[ci]]
        for ri, (_key, _tt, _pp, _col, _lbl) in enumerate(PLOT_CONDS):
            _ax = axes_cells[ri][ci]
            _ct = _gc(_tt, _pp)
            if _ct:
                _tc = _trial_tc(_cid, _ct, clusters_s, beh_s, trials_s)
                if _tc is not None:
                    _pfr(_ax, _tc, bs=BIN_SIZE, tl=TRACK_LEN, p=95, cmap=cmap_parula)
            _ax.set_yticks([])
            _ax.spines[['top','right','left']].set_visible(False)
            _ax.tick_params(axis='x', labelsize=5.5)
            if ri < 3: _ax.set_xticklabels([])
            if ri == 0: _ax.set_title(f'Cell {ci+1}', fontsize=7.5, fontweight='bold')
        axes_cells[3][ci].set_xlabel('Position (cm)', fontsize=6)

    # col 6: decoded position heatmap (jet)
    for ri, (_key, _tt, _pp, _col, _lbl) in enumerate(PLOT_CONDS):
        _ax = ax_dec[ri]
        _dm = dec_map_s.get(_key)
        if _dm is not None:
            _nt, _nb = _dm.shape
            _ye = np.linspace(0, TRACK_LEN, _nb+1)
            _xe = np.arange(1, _nt+2)
            _X, _Y = np.meshgrid(_xe, _ye)
            _h = _ax.pcolormesh(_Y, _X, _dm.T, shading='auto', cmap='jet',
                                 vmin=0, vmax=TRACK_LEN)
            _h.set_rasterized(True)
            _ax.set_xlim(0, TRACK_LEN); _ax.set_ylim(1, _nt+1); _ax.invert_yaxis()
        else:
            _ax.text(0.5, 0.5, 'insufficient\ntrials', ha='center', va='center',
                     transform=_ax.transAxes, fontsize=6, color='gray')
        _ax.set_yticks([])
        _ax.spines[['top','right','left']].set_visible(False)
        _ax.tick_params(axis='x', labelsize=5.5)
        if ri < 3: _ax.set_xticklabels([])
    ax_dec[3].set_xlabel('Actual pos. (cm)', fontsize=6)
    ax_dec[0].set_title('Decoded pos.', fontsize=7.5, fontweight='bold')

    fig.suptitle(
        f'M{_mouse}D{_day} — Bayesian Poisson decoding pipeline  ({len(mec_s)} MEC cells)',
        fontsize=10, fontweight='bold', y=0.995)
    plt.savefig(f'vr_pipeline_M{_mouse}D{_day}.pdf', bbox_inches='tight', dpi=150)
    plt.show()
    print('done')


---
## Population decoding performance

Mean circular decoding error across all sessions and trial conditions.

In [ ]:
sess_err = (df_decode.groupby(['mouse','day'])['error_cm']
            .mean().reset_index().rename(columns={'error_cm': 'mean_err'}))
dec_order = ['bayesian_poisson']   # single decoder

fig, axes = plt.subplots(1, 2, figsize=(11, 5), gridspec_kw=dict(wspace=0.38))

# ── Violin + jitter: per-session mean error ────────────────────────────────
ax = axes[0]
vals = sess_err['mean_err'].dropna().values
parts = ax.violinplot(vals, positions=[0], widths=0.6,
                      showmedians=True, showextrema=False)
for pc in parts['bodies']:
    pc.set_facecolor(DECODER_COLOUR); pc.set_alpha(0.45)
parts['cmedians'].set_color(DECODER_COLOUR); parts['cmedians'].set_linewidth(2.5)
rng = np.random.default_rng(0)
ax.scatter(rng.uniform(-0.15, 0.15, len(vals)), vals,
           s=10, color=DECODER_COLOUR, alpha=0.70, lw=0)
ax.axhline(CHANCE_CM, color='#90A4AE', lw=1.0, ls='--',
           label=f'Chance ({CHANCE_CM:.0f} cm)')
ax.set_xticks([0]); ax.set_xticklabels(['Bayesian Poisson'], fontsize=8)
ax.set_ylabel('Mean circular decoding error (cm)', fontsize=9)
ax.set_title(f'Per-session error  (n={len(vals)} sessions)\nMedian = {np.median(vals):.1f} cm', fontsize=9, fontweight='bold')
ax.legend(fontsize=7.5, frameon=False)
ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=7)

# ── CDF ────────────────────────────────────────────────────────────────────
ax = axes[1]
all_err = np.sort(df_decode['error_cm'].dropna().values)
cdf = np.arange(1, len(all_err)+1) / len(all_err)
ax.plot(all_err, cdf, color=DECODER_COLOUR, lw=1.8, label='Bayesian Poisson')
ax.axvline(CHANCE_CM, color='#90A4AE', lw=1.0, ls='--', label=f'Chance ({CHANCE_CM:.0f} cm)')
ax.set_xlabel('Circular decoding error (cm)', fontsize=9)
ax.set_ylabel('Cumulative fraction', fontsize=9)
ax.set_title('Error CDF — all observations', fontsize=9, fontweight='bold')
ax.legend(fontsize=7.5, frameon=False)
ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=7)

fig.suptitle('Bayesian Poisson — MEC VR position decoding',
             fontsize=10, fontweight='bold')
plt.savefig('vr_decode_summary.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── Per-session n_cells vs mean decoding error ────────────────────────────
sess_n = (df_decode.groupby(['mouse','day','n_cells'])['error_cm']
          .mean().reset_index().rename(columns={'error_cm':'mean_err'})).dropna()

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(sess_n['n_cells'], sess_n['mean_err'],
           color=DECODER_COLOUR, s=45, alpha=0.80, lw=0)
if len(sess_n) > 2:
    m, b = np.polyfit(sess_n['n_cells'], sess_n['mean_err'], 1)
    xr = np.array([sess_n['n_cells'].min(), sess_n['n_cells'].max()])
    ax.plot(xr, m * xr + b, color=DECODER_COLOUR, lw=1.5, ls='--', alpha=0.7)
    r, p = spearmanr(sess_n['n_cells'], sess_n['mean_err'])
    sig  = ' *' if p < 0.05 else ''
    ax.set_title(f'Spearman ρ = {r:.2f}{sig}  (p = {p:.3f})  n = {len(sess_n)} sessions',
                 fontsize=8.5, fontweight='bold', color=DECODER_COLOUR)
ax.axhline(CHANCE_CM, color='#90A4AE', lw=0.8, ls='--', label=f'Chance ({CHANCE_CM:.0f} cm)')
ax.set_xlabel('N MEC principal cells', fontsize=9)
ax.set_ylabel('Mean circular decoding error (cm)', fontsize=9)
ax.legend(fontsize=7.5, frameon=False)
ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=7)
fig.suptitle('N cells vs decoding error — per session', fontsize=10, fontweight='bold')
plt.savefig('vr_decode_ncells_vs_error.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── Session ML position vs decoding error — single-shank sessions only ─────
# For multi-shank sessions the mean ML position is not meaningful (cells span
# a range of shanks).  Restricting to single-shank sessions (ML spread of
# recorded cells < MIN_ML_SPREAD_UM) gives a clean session-level ML coordinate:
# every cell on the shank sits at roughly the same ML position, so the mean
# is a valid proxy for where the probe was implanted.
# This is the complement of the within-session ablation (which uses multi-shank).

_clf = pd.read_csv(CLASSIFICATIONS_CSV,
                   usecols=['mouse','day','cluster_id',
                            'brain_region','firing_rate_VR','SC_x'])
_clf['mouse'] = _clf['mouse'].astype(int)
_clf['day']   = _clf['day'].astype(int)

# Per-session: ML spread and mean for MEC principal cells
sess_ml_rows = []
for (mouse, day), grp in _clf.groupby(['mouse','day']):
    mec = grp[
        grp['brain_region'].str.startswith('ENTm', na=False) &
        (grp['firing_rate_VR'] < PRINCIPAL_MAX_FR) &
        grp['SC_x'].notna()
    ]
    if len(mec) < MIN_MEC_CELLS:
        continue
    ml_abs  = mec['SC_x'].abs()
    ml_spread = ml_abs.max() - ml_abs.min()
    # Single-shank only: spread below threshold means all cells share one ML position
    if ml_spread >= MIN_ML_SPREAD_UM:
        continue
    sess_ml_rows.append(dict(mouse=mouse, day=day,
                             mean_ml=ml_abs.mean(),
                             n_cells=len(mec)))
df_sess_ml = pd.DataFrame(sess_ml_rows)

sess_err = (df_decode.groupby(['mouse','day'])['error_cm']
            .mean().reset_index().rename(columns={'error_cm':'mean_err'}))
sess_ml_err = sess_err.merge(df_sess_ml, on=['mouse','day'], how='inner').dropna()

print(f'Single-shank sessions included: {len(sess_ml_err)}')

fig, ax = plt.subplots(figsize=(6, 5))

sc = ax.scatter(sess_ml_err['mean_ml'], sess_ml_err['mean_err'],
                c=sess_ml_err['n_cells'], cmap='YlOrRd_r',
                s=55, alpha=0.85, lw=0.5, edgecolors='k',
                vmin=sess_ml_err['n_cells'].min(),
                vmax=sess_ml_err['n_cells'].max())
plt.colorbar(sc, ax=ax, label='N MEC principal cells', pad=0.02)

if len(sess_ml_err) > 2:
    m, b = np.polyfit(sess_ml_err['mean_ml'], sess_ml_err['mean_err'], 1)
    xr = np.array([sess_ml_err['mean_ml'].min(), sess_ml_err['mean_ml'].max()])
    ax.plot(xr, m * xr + b, color=DECODER_COLOUR, lw=1.6, ls='--', alpha=0.8)
    r, p = spearmanr(sess_ml_err['mean_ml'], sess_ml_err['mean_err'])
    sig  = ' *' if p < 0.05 else ''
    title = f'Spearman ρ = {r:.2f}{sig}  (p = {p:.3f})  n = {len(sess_ml_err)} sessions'
else:
    title = 'Too few single-shank sessions'

ax.axhline(CHANCE_CM, color='#90A4AE', lw=0.8, ls='--',
           label=f'Chance ({CHANCE_CM:.0f} cm)')
ax.set_xlabel('Session ML position (µm, |SC_x|)', fontsize=9)
ax.set_ylabel('Mean circular decoding error (cm)', fontsize=9)
ax.set_title(title, fontsize=8.5, fontweight='bold', color=DECODER_COLOUR)
ax.legend(fontsize=7.5, frameon=False)
ax.spines[['top','right']].set_visible(False)
ax.tick_params(labelsize=7)

fig.suptitle('Single-shank sessions: ML position vs decoding error\n(colour = N cells; multi-shank sessions excluded)',
             fontsize=10, fontweight='bold')
plt.savefig('vr_decode_ml_vs_error.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
def _track_error_lines(ax, df, title, region_col):
    if df is None or len(df) == 0 or 'actual_pos' not in df.columns:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                transform=ax.transAxes, fontsize=9, color='gray')
        ax.set_title(title, fontsize=9, fontweight='bold', color=region_col)
        ax.spines[['top','right']].set_visible(False)
        return
    df = df.copy()
    df['pos_bin'] = np.digitize(df['actual_pos'].values, BIN_EDGES[1:-1])
    ax.axhline(CHANCE_CM, color='#90A4AE', lw=0.8, ls='--',
               label=f'Chance ({CHANCE_CM:.0f} cm)')
    for key, ttype, perf, col in TRIAL_GROUPS:
        grp = df if ttype is None else df[(df['trial_type']==ttype)&(df['performance']==perf)]
        if len(grp) == 0:
            continue
        _s = grp.groupby(['mouse','day','pos_bin'])['error_cm'].mean().reset_index()
        g  = _s.groupby('pos_bin')['error_cm'].agg(['mean','sem']).reindex(range(N_BINS))
        ax.fill_between(BIN_CENTRES, g['mean']-g['sem'], g['mean']+g['sem'],
                        alpha=0.12, color=col, lw=0)
        ax.plot(BIN_CENTRES, g['mean'], color=col, lw=1.5, label=key)
    n_sess = df.groupby(['mouse','day']).ngroups
    ax.set_xlim(0, TRACK_LEN)
    ax.set_xlabel('Actual position (cm)', fontsize=9)
    ax.set_ylabel('Circular decoding error (cm)', fontsize=9)
    ax.set_title(f'{title}  (n={n_sess} sessions)', fontsize=9, fontweight='bold',
                 color=region_col)
    ax.legend(fontsize=6.5, frameon=False)
    ax.spines[['top','right']].set_visible(False)
    ax.tick_params(labelsize=7)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True, gridspec_kw=dict(wspace=0.08))

_track_error_lines(axes[0], df_decode, 'MEC principal cells', DECODER_COLOUR)
_track_error_lines(axes[1], df_vis,    'VIS principal cells', VIS_COLOUR)

fig.suptitle('Decoding error across VR track — Bayesian Poisson (mean ± SEM across sessions)',
             fontsize=10, fontweight='bold')
plt.savefig('vr_decode_error_by_position.pdf', bbox_inches='tight', dpi=150)
plt.show()

### Hit : run decoding error ratio

`error_hit / error_run` at each position along the track.
Values **< 1** indicate the decoder is more accurate on successful (hit) trials;
values **> 1** indicate worse decoding on hits than on run-through trials.
Computed per session, then averaged across sessions (± SEM).

In [ ]:
# ── Build per-session hit:run ratio at each position bin ──────────────────
if 'pos_bin' not in df_decode.columns or df_decode['pos_bin'].dtype == float:
    df_decode['pos_bin'] = np.digitize(df_decode['actual_pos'].values, BIN_EDGES[1:-1])

sess_pos = (df_decode
            .groupby(['mouse','day','trial_type','performance','pos_bin'])
            ['error_cm'].mean()
            .reset_index())

ratio_rows = []
for (mouse, day), grp in sess_pos.groupby(['mouse','day']):
    for ttype in ['b', 'nb']:
        hit = (grp[(grp['trial_type']==ttype) & (grp['performance']=='hit')]
               .set_index('pos_bin')['error_cm'])
        run = (grp[(grp['trial_type']==ttype) & (grp['performance']=='run')]
               .set_index('pos_bin')['error_cm'])
        common = hit.index.intersection(run.index)
        for bi in common:
            if run[bi] > 0:
                ratio_rows.append(dict(
                    mouse=mouse, day=day,
                    trial_type=ttype, pos_bin=int(bi),
                    ratio=hit[bi] / run[bi],
                ))

df_ratio = pd.DataFrame(ratio_rows)
print(f'Ratio rows: {len(df_ratio)}')

# ── Figure 1: ratio vs position ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw=dict(wspace=0.35))

for ax, (ttype, label) in zip(axes, [('b', 'Beaconed'), ('nb', 'Non-beaconed')]):
    ax.axhline(1.0, color='k', lw=1.0, ls='--', label='No difference (ratio = 1)')
    sub = df_ratio[df_ratio['trial_type'] == ttype]
    if len(sub) > 0:
        _sess_r = sub.groupby(['mouse','day','pos_bin'])['ratio'].mean().reset_index()
        avg = (_sess_r.groupby('pos_bin')['ratio']
                      .agg(['mean','sem'])
                      .reindex(range(N_BINS)))
        ax.fill_between(BIN_CENTRES,
                        avg['mean'] - avg['sem'], avg['mean'] + avg['sem'],
                        alpha=0.15, color=DECODER_COLOUR, lw=0)
        ax.plot(BIN_CENTRES, avg['mean'], color=DECODER_COLOUR, lw=1.8)
    ax.set_xlim(0, TRACK_LEN)
    ax.set_xlabel('Actual position (cm)', fontsize=9)
    ax.set_ylabel('Hit : Run error ratio', fontsize=9)
    ax.set_title(f'{label} — hit:run error ratio  (< 1 = better decoding on hit trials)', fontsize=9, fontweight='bold')
    ax.legend(fontsize=7.5, frameon=False)
    ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=7)

fig.suptitle('Hit : Run decoding error ratio across VR track (mean ± SEM across sessions)', fontsize=10, fontweight='bold')
plt.savefig('vr_decode_hit_run_ratio_track.pdf', bbox_inches='tight', dpi=150)
plt.show()

# ── Figure 2: per-session mean ratio — violin ─────────────────────────────
sess_ratio_mean = (df_ratio.groupby(['mouse','day','trial_type'])['ratio']
                   .mean().reset_index())

fig, axes = plt.subplots(1, 2, figsize=(9, 5), gridspec_kw=dict(wspace=0.35))

for ax, (ttype, label) in zip(axes, [('b', 'Beaconed'), ('nb', 'Non-beaconed')]):
    ax.axhline(1.0, color='k', lw=1.0, ls='--')
    vals = sess_ratio_mean[sess_ratio_mean['trial_type'] == ttype]['ratio'].dropna().values
    if len(vals) == 0:
        continue
    parts = ax.violinplot(vals, positions=[0], widths=0.6,
                          showmedians=True, showextrema=False)
    for pc in parts['bodies']:
        pc.set_facecolor(DECODER_COLOUR); pc.set_alpha(0.45)
    parts['cmedians'].set_color(DECODER_COLOUR); parts['cmedians'].set_linewidth(2.2)
    rng = np.random.default_rng(0)
    ax.scatter(rng.uniform(-0.15, 0.15, len(vals)), vals,
               s=9, color=DECODER_COLOUR, alpha=0.70, lw=0)
    ax.set_xticks([0]); ax.set_xticklabels(['Bayesian Poisson'], fontsize=8)
    ax.set_ylabel('Mean hit : run error ratio', fontsize=9)
    ax.set_title(f'{label}  (median = {np.median(vals):.2f})', fontsize=9, fontweight='bold')
    ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=7)

fig.suptitle('Hit : Run decoding error ratio — per session mean (averaged across track positions)', fontsize=10, fontweight='bold')
plt.savefig('vr_decode_hit_run_ratio_summary.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── Per-condition decoding error ──────────────────────────────────────────
sess_cond = (df_decode.groupby(['mouse','day','trial_type','performance'])
             ['error_cm'].mean().reset_index())
sess_cond['group'] = sess_cond['trial_type'] + '_' + sess_cond['performance']

present_groups = [k for k, *_ in TRIAL_GROUPS[1:]
                  if k in sess_cond['group'].values]
COND_COLOUR_MAP = {k: c for k, _, _, c in TRIAL_GROUPS}

fig, ax = plt.subplots(figsize=(5, 5))
ax.axhline(CHANCE_CM, color='#90A4AE', lw=0.8, ls='--', label=f'Chance ({CHANCE_CM:.0f} cm)')
for xi, grp in enumerate(present_groups):
    vals = sess_cond[sess_cond['group'] == grp]['error_cm'].dropna().values
    if len(vals) == 0:
        continue
    col_grp = COND_COLOUR_MAP.get(grp, DECODER_COLOUR)
    sem = vals.std(ddof=1) / np.sqrt(len(vals))
    ax.bar(xi, vals.mean(), color=col_grp, alpha=0.80, lw=0)
    ax.errorbar(xi, vals.mean(), sem, color='k', lw=1.2, capsize=3, zorder=5)

ax.set_xticks(range(len(present_groups)))
ax.set_xticklabels(present_groups, fontsize=8, rotation=35, ha='right')
ax.set_ylabel('Mean circular decoding error (cm)', fontsize=9)
ax.legend(fontsize=7.5, frameon=False)
ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=7)
fig.suptitle('Per-condition decoding error — Bayesian Poisson (mean ± SEM across sessions)',
             fontsize=10, fontweight='bold')
plt.savefig('vr_decode_conditions.pdf', bbox_inches='tight', dpi=150)
plt.show()

---
## Condition-specificity of the position code

The Bayesian decoder is trained on **each trial condition separately** (even-indexed trials
of that type only) and tested on **all conditions** (odd-indexed trials).

The result is a **train × test error matrix**:
- **Rows** = what condition the tuning curves were estimated from
- **Columns** = what condition was decoded at test time
- Low values on the diagonal = condition-specific position codes
- Flat rows = a general position representation that generalises across conditions

Minimum `MIN_COND_TRAIN_OBS` training observations required per condition; sessions
with too few trials for a given condition are excluded from that row.

In [ ]:
# ── Train × test decoding error matrix ────────────────────────────────────
df_cond_train['test_condition'] = (df_cond_train['trial_type'] + '_' +
                                    df_cond_train['performance'])

train_order = [tc for tc, *_ in TRAIN_CONDITIONS]
test_order  = [k for k, *_ in TRIAL_GROUPS[1:]]   # b_hit, b_run, nb_hit, nb_run

# Per-session mean error for each (train_condition, test_condition) pair
sess_mat = (df_cond_train
            .groupby(['mouse','day','train_condition','test_condition'])['error_cm']
            .mean().reset_index())

# Population mean across sessions
mat_mean = (sess_mat.groupby(['train_condition','test_condition'])['error_cm']
            .mean().unstack('test_condition')
            .reindex(index=train_order, columns=test_order))

mat_sem  = (sess_mat.groupby(['train_condition','test_condition'])['error_cm']
            .sem().unstack('test_condition')
            .reindex(index=train_order, columns=test_order))

n_sess   = (sess_mat.groupby(['train_condition','test_condition'])
            .size().unstack('test_condition')
            .reindex(index=train_order, columns=test_order))

fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw=dict(wspace=0.45))

# ── Heatmap ────────────────────────────────────────────────────────────────
ax = axes[0]
im = ax.imshow(mat_mean.values, cmap='RdYlGn_r', aspect='auto',
               vmin=0, vmax=CHANCE_CM)
plt.colorbar(im, ax=ax, label='Mean circular error (cm)', pad=0.02)
ax.set_xticks(range(len(test_order)));  ax.set_xticklabels(test_order,  fontsize=8, rotation=35, ha='right')
ax.set_yticks(range(len(train_order))); ax.set_yticklabels(train_order, fontsize=8)
ax.set_xlabel('Test condition', fontsize=9)
ax.set_ylabel('Train condition', fontsize=9)
ax.set_title('Mean decoding error (cm)\n(red = worse, green = better)', fontsize=9, fontweight='bold')
# Annotate cells with mean ± sem
for ri, tr in enumerate(train_order):
    for ci, te in enumerate(test_order):
        mu  = mat_mean.loc[tr, te] if tr in mat_mean.index and te in mat_mean.columns else np.nan
        ns  = n_sess.loc[tr, te]   if tr in n_sess.index   and te in n_sess.columns   else 0
        if np.isfinite(mu):
            ax.text(ci, ri, f'{mu:.1f}\n(n={int(ns)})',
                    ha='center', va='center', fontsize=6.5,
                    color='white' if mu > CHANCE_CM * 0.6 else 'black')

# ── Bar chart: each train condition, bars = test conditions ───────────────
ax = axes[1]
COND_COLOUR_MAP = {k: c for k, _, _, c in TRIAL_GROUPS}
x = np.arange(len(train_order))
width = 0.18
offsets = np.linspace(-(len(test_order)-1)/2, (len(test_order)-1)/2, len(test_order)) * width

ax.axhline(CHANCE_CM, color='#90A4AE', lw=0.8, ls='--', label=f'Chance ({CHANCE_CM:.0f} cm)')
for oi, te in enumerate(test_order):
    col = COND_COLOUR_MAP.get(te, '#455A64')
    mu_vals = [mat_mean.loc[tr, te] if (tr in mat_mean.index and te in mat_mean.columns
               and np.isfinite(mat_mean.loc[tr, te])) else np.nan for tr in train_order]
    se_vals = [mat_sem.loc[tr, te]  if (tr in mat_sem.index  and te in mat_sem.columns
               and np.isfinite(mat_sem.loc[tr, te]))  else 0   for tr in train_order]
    ax.bar(x + offsets[oi], mu_vals, width, color=col, alpha=0.80, lw=0, label=te)
    ax.errorbar(x + offsets[oi], mu_vals, se_vals,
                fmt='none', color='k', lw=1.0, capsize=2)

ax.set_xticks(x); ax.set_xticklabels(train_order, fontsize=8, rotation=35, ha='right')
ax.set_ylabel('Mean circular decoding error (cm)', fontsize=9)
ax.set_xlabel('Train condition', fontsize=9)
ax.set_title('Decoding error by train × test condition', fontsize=9, fontweight='bold')
ax.legend(fontsize=7, frameon=False, title='Test condition', title_fontsize=7)
ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=7)

fig.suptitle('Position code condition-specificity — train × test decoding error matrix',
             fontsize=10, fontweight='bold')
plt.savefig('vr_decode_train_test_matrix.pdf', bbox_inches='tight', dpi=150)
plt.show()

print('\nMean error matrix (cm):')
print(mat_mean.round(1).to_string())

---
## ML ablation — medial vs lateral contribution

Cells are sorted by `abs(SC_x)` (ML position) and progressively removed from one end.
The decoder is **refit** at every step, so curves reflect the true encoding capacity
of each sub-population.

**Multi-shank sessions only.** Sessions where the ML spread of recorded cells is
< `MIN_ML_SPREAD_UM` (default 200 µm) are excluded — single-shank recordings have all
cells at approximately the same ML position, making the medial → lateral ordering
uninterpretable.

Decoding error at each ablation step is broken down by **trial type × outcome**
(beaconed/non-beaconed × hit/run) to show whether the ML gradient differs across conditions.

In [ ]:
N_FRAC_BINS  = 20
frac_edges   = np.linspace(0, 1, N_FRAC_BINS + 1)
frac_centres = 0.5 * (frac_edges[:-1] + frac_edges[1:])
df_ablation['frac_bin'] = pd.cut(df_ablation['frac_kept'],
                                   bins=frac_edges, labels=False, include_lowest=True)

abl_decs   = [d for d in ABLATION_DECODERS if d in df_ablation['decoder'].unique()]
DIR_LABELS = {'medial_first': 'Remove medial first', 'lateral_first': 'Remove lateral first'}

# Condition colours (reuse TRIAL_GROUPS palette + grey for 'all')
COND_COLS = {k: c for k, *_, c in TRIAL_GROUPS}
COND_COLS['all'] = '#455A64'
COND_LW   = {'all': 2.2, 'b_hit': 1.6, 'b_run': 1.6, 'nb_hit': 1.6, 'nb_run': 1.6}
COND_LS   = {'all': '-',  'b_hit': '-',  'b_run': '--', 'nb_hit': '-',  'nb_run': '--'}
COND_ORDER = ['all', 'b_hit', 'b_run', 'nb_hit', 'nb_run']

# ── Layout: 2 rows (directions) × n_decoders columns ─────────────────────
n_dec = len(abl_decs)
fig, axes = plt.subplots(2, n_dec,
                          figsize=(6 * n_dec, 9),
                          gridspec_kw=dict(hspace=0.38, wspace=0.30),
                          squeeze=False)

for col_i, dec in enumerate(abl_decs):
    col_dec = DECODER_COLOUR
    for row_i, direction in enumerate(['medial_first', 'lateral_first']):
        ax  = axes[row_i, col_i]
        sub = df_ablation[
            (df_ablation['decoder']   == dec) &
            (df_ablation['direction'] == direction)
        ]
        ax.axhline(CHANCE_CM, color='#90A4AE', lw=0.8, ls=':', label='Chance')
        for cond in COND_ORDER:
            d = sub[sub['condition'] == cond]
            if len(d) == 0:
                continue
            binned = (d.groupby(['frac_bin','mouse','day'])['mean_error_cm']
                       .mean().reset_index())
            avg = binned.groupby('frac_bin')['mean_error_cm'].agg(['mean','sem'])
            x   = frac_centres[avg.index]
            col = COND_COLS.get(cond, '#455A64')
            ax.fill_between(x, avg['mean']-avg['sem'], avg['mean']+avg['sem'],
                            alpha=0.10, color=col, lw=0)
            ax.plot(x, avg['mean'], color=col,
                    lw=COND_LW.get(cond,1.5), ls=COND_LS.get(cond,'-'),
                    label=cond)
        ax.invert_xaxis()
        ax.set_xlabel('Fraction of cells remaining', fontsize=8)
        ax.set_ylabel('Mean circular error (cm)', fontsize=8)
        ax.set_title(f'{dec.replace("_"," ")} — {DIR_LABELS[direction]}', fontsize=8.5, fontweight='bold', color=col_dec)
        ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=7)
        if row_i == 0 and col_i == 0:
            ax.legend(fontsize=7, frameon=False)

fig.suptitle('ML ablation by trial type & outcome (mean ± SEM across sessions  |  x inverted: all → no cells)', fontsize=10, fontweight='bold')
plt.savefig('vr_decode_ablation_by_condition.pdf', bbox_inches='tight', dpi=150)
plt.show()